# Self-Critiquing QA Workflow with Routing and Refinement

This upgrades the single-node "ask an LLM a question" graph into a multi-stage
LangGraph workflow that:

1. **Classifies** the incoming question (`factual`, `math`, `creative`, or `opinion`).
2. **Routes** it to a generation node whose prompt is tailored to that category.
3. **Critiques** the generated answer with a strict grader model that returns a
   structured score (1-10) and written feedback.
4. **Refines** the answer using that feedback and sends it back for another
   critique pass, looping until the score is high enough or a retry limit is hit.
5. **Finalizes** the best answer once the loop exits.

The point of the example is to show the LangGraph features a single straight-line
chain can't: conditional branching (`add_conditional_edges`), multiple branches
converging back into one node, and a cycle in the graph that terminates itself
via state (an `attempts` counter) instead of running forever.


In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict, Literal
from pydantic import BaseModel, Field
from dotenv import load_dotenv


In [ ]:
load_dotenv()  # Load environment variables from .env file


## 1. State schema

`LLMState` is the shared dict every node reads from and writes back to. Adding
fields here (`category`, `score`, `attempts`, ...) is what lets nodes downstream
know what earlier nodes decided, and lets the conditional edges make routing
decisions based on that history.


In [ ]:
class LLMState(TypedDict):
    question: str        # the user's original question
    category: str        # set by classify_node: factual | math | creative | opinion
    answer: str           # current draft answer, overwritten on each refine pass
    critique: str          # feedback from the last critique pass
    score: int            # 1-10 quality score from the last critique pass
    attempts: int         # how many refine passes have happened so far
    final_answer: str      # set once the loop exits


## 2. Models

Two model configurations, same underlying model:

- `precise_model` (temperature `0`) for tasks that need consistent, structured
  output: classifying the question and grading the answer.
- `generation_model` (temperature `0.7`) for actually writing answers, where a
  bit of variety is fine (and desirable for the `creative` category).

Both are wrapped with `.with_structured_output(...)` where we need a Pydantic
object back instead of raw text — that removes the need to hand-parse the
model's reply to pull out a category or a numeric score.


In [ ]:
MODEL_NAME = "gemini-3.8-flash"

precise_model = ChatGoogleGenerativeAI(
    model=MODEL_NAME,
    temperature=0,
    max_output_tokens=256,
)

generation_model = ChatGoogleGenerativeAI(
    model=MODEL_NAME,
    temperature=0.7,
    max_output_tokens=300,
)


## 3. Classification node

`classify_node` asks the precise model to bucket the question into one of four
categories using a Pydantic schema (`QuestionCategory`), so the result is
always a valid literal rather than free text we'd have to parse and validate
ourselves.


In [ ]:
class QuestionCategory(BaseModel):
    category: Literal["factual", "math", "creative", "opinion"] = Field(
        description="The single best category for the question."
    )
    reasoning: str = Field(description="One short sentence justifying the category.")


def classify_node(state: LLMState) -> LLMState:
    classifier = precise_model.with_structured_output(QuestionCategory)
    result = classifier.invoke(
        "Classify the following question into exactly one category: "
        "factual, math, creative, or opinion.\n\n"
        f"Question: {state['question']}"
    )
    state["category"] = result.category
    return state


## 4. Category-specific generation nodes

Rather than one generic prompt for every question, each category gets a node
with a prompt suited to it: factual answers stay short and grounded, math
answers are told to show their work, creative answers use the higher-
temperature model, and opinion answers are told to present a balanced view.
All four converge on the same next step (`critique`), so the graph fans out
and back in.


In [ ]:
def generate_factual(state: LLMState) -> LLMState:
    prompt = (
        "Answer the following factual question accurately and concisely, "
        f"in 1-3 sentences: {state['question']}"
    )
    state["answer"] = generation_model.invoke(prompt).content
    return state


def generate_math(state: LLMState) -> LLMState:
    prompt = (
        "Solve the following math problem. Show your reasoning step by step, "
        f"then give a final numeric answer on its own line: {state['question']}"
    )
    state["answer"] = generation_model.invoke(prompt).content
    return state


def generate_creative(state: LLMState) -> LLMState:
    prompt = f"Respond creatively and imaginatively to: {state['question']}"
    state["answer"] = generation_model.invoke(prompt).content
    return state


def generate_opinion(state: LLMState) -> LLMState:
    prompt = (
        "Give a balanced, well-reasoned perspective on the following, "
        f"briefly acknowledging at least one other viewpoint: {state['question']}"
    )
    state["answer"] = generation_model.invoke(prompt).content
    return state


def route_by_category(state: LLMState) -> str:
    """Conditional-edge function: returns the name of the next node."""
    return state["category"]


## 5. Critique node

`critique_node` grades the current `answer` against the original `question`
using the same structured-output pattern, this time returning a `CritiqueResult`
with an integer `score` and written `feedback`. This is the node the loop
pivots on.


In [ ]:
class CritiqueResult(BaseModel):
    score: int = Field(ge=1, le=10, description="Quality score, 1 (poor) to 10 (excellent).")
    feedback: str = Field(description="Specific, actionable feedback for improving the answer.")


def critique_node(state: LLMState) -> LLMState:
    critic = precise_model.with_structured_output(CritiqueResult)
    result = critic.invoke(
        "Grade the following answer for accuracy, completeness, and clarity. "
        "Be strict.\n\n"
        f"Question: {state['question']}\n"
        f"Answer: {state['answer']}"
    )
    state["score"] = result.score
    state["critique"] = result.feedback
    return state


## 6. Refine node and the loop's exit condition

`refine_node` rewrites the answer using the critique's feedback and bumps
`attempts`. `decide_after_critique` is the conditional-edge function attached
to `critique`: if the score is good enough, or the retry budget
(`MAX_ATTEMPTS`) is used up, it routes to `finalize`; otherwise it sends the
graph back to `refine` → `critique` again. The `attempts` counter is what
keeps this from looping forever on a question the model can't answer well.


In [ ]:
MAX_ATTEMPTS = 2


def refine_node(state: LLMState) -> LLMState:
    state["attempts"] = state.get("attempts", 0) + 1
    prompt = (
        "Improve the answer below using the critique. Keep it concise.\n\n"
        f"Question: {state['question']}\n"
        f"Current answer: {state['answer']}\n"
        f"Critique: {state['critique']}\n\n"
        "Write only the improved answer."
    )
    state["answer"] = generation_model.invoke(prompt).content
    return state


def decide_after_critique(state: LLMState) -> str:
    """Conditional-edge function: 'finalize' to exit the loop, 'refine' to continue it."""
    if state["score"] >= 8 or state.get("attempts", 0) >= MAX_ATTEMPTS:
        return "finalize"
    return "refine"


def finalize_node(state: LLMState) -> LLMState:
    state["final_answer"] = state["answer"]
    return state


## 7. Assembling the graph

- `add_conditional_edges("classify", route_by_category, {...})` fans out to the
  four generation nodes based on `state["category"]`.
- All four generation nodes have a plain `add_edge(..., "critique")`, so they
  converge back into one node regardless of which branch ran.
- `add_conditional_edges("critique", decide_after_critique, {...})` is the loop:
  it either exits to `finalize` or sends control back to `refine`, which in turn
  always edges back to `critique` — that `refine → critique` edge is what makes
  this a cycle instead of a straight line.


In [ ]:
graph = StateGraph(LLMState)

# nodes
graph.add_node("classify", classify_node)
graph.add_node("generate_factual", generate_factual)
graph.add_node("generate_math", generate_math)
graph.add_node("generate_creative", generate_creative)
graph.add_node("generate_opinion", generate_opinion)
graph.add_node("critique", critique_node)
graph.add_node("refine", refine_node)
graph.add_node("finalize", finalize_node)

# entry
graph.add_edge(START, "classify")

# fan-out: route to a generation node based on category
graph.add_conditional_edges(
    "classify",
    route_by_category,
    {
        "factual": "generate_factual",
        "math": "generate_math",
        "creative": "generate_creative",
        "opinion": "generate_opinion",
    },
)

# fan-in: every generation node converges on critique
graph.add_edge("generate_factual", "critique")
graph.add_edge("generate_math", "critique")
graph.add_edge("generate_creative", "critique")
graph.add_edge("generate_opinion", "critique")

# loop: critique either exits to finalize or bounces to refine, which loops back
graph.add_conditional_edges(
    "critique",
    decide_after_critique,
    {
        "refine": "refine",
        "finalize": "finalize",
    },
)
graph.add_edge("refine", "critique")

graph.add_edge("finalize", END)

workflow = graph.compile()


## 8. Visualizing the graph (optional)

Renders the compiled graph as a Mermaid diagram so you can see the fan-out /
fan-in / loop structure described above. Requires network access and the
`grandalf`/mermaid rendering extras that ship with `langgraph`; this is
wrapped in a `try` so the notebook still runs without them.


In [ ]:
try:
    from IPython.display import Image, display

    display(Image(workflow.get_graph().draw_mermaid_png()))
except Exception as e:
    print("Skipping visualization:", e)


## 9. Running it

Three questions, one per category, so you can see the routing pick a different
generation node each time and the loop run a different number of `attempts`
depending on how the critique scores the first draft.


In [ ]:
for question in [
    "What is the capital of France?",
    "If a train travels 60 mph for 2.5 hours, how far does it go?",
    "Write a two-line poem about autumn leaves.",
]:
    result = workflow.invoke({"question": question, "attempts": 0})
    print(f"Q: {question}")
    print(f"category={result['category']}  score={result['score']}  attempts={result['attempts']}")
    print(f"A: {result['final_answer']}")
    print("-" * 60)
